# Dataset color distributions

Research question: how do Food-101 and COCO differ in physical Lab hue and
chroma, and how might those differences confound model comparisons?
Historical ClearML artifacts are considered lost. This notebook contains
the analysis protocol, not recovered measurements. No analysis has been run.
Start Jupyter from the repository root using `scripts/lab.sh`.

## Published context

The 2025 color-diversity abstract discusses warm hue bias in Food-101 and COCO.
Its numerical example gives a red-sector fraction of approximately 0.269 for
originals, but does not identify a dataset/checkpoint pair for each value.
It provides no full histogram that could be reconstructed here.
Read the [published-results record](../reports/published-results.md) for the
source and limitations. The optional analysis below remains unexecuted;
new measurements are outside this revision's scope.

In [ ]:
from pathlib import Path

import torch
from torch.utils.data import DataLoader

from chromatica.config import load_config
from chromatica.datasets.dataset import DirectoryImageDataset
from chromatica.metrics.hue import compute_histogram
from chromatica.metrics.palettes import compute_top_colors_modes
from chromatica.visualization.color_diversity import (
    plot_histogram,
    plot_top_colors_palette,
)

ROOT = Path.cwd()
RUN_ANALYSIS = False
CONFIG_PATHS = [
    ROOT / "configs/food101_unet.toml",
    ROOT / "configs/coco_unet.toml",
]

Use the same image size, bilinear antialiased resizing, affine Lab encoding,
hue bins, and chroma threshold for both datasets. This protocol counts all
pixels; palette candidates are modes of physical a*/b*, not normalized values.
Hue percentages describe eligible pixels. Report their counts alongside total
pixels because the excluded low-chroma population can differ between datasets.

In [ ]:
if RUN_ANALYSIS:
    for config_path in CONFIG_PATHS:
        config = load_config(config_path)
        dataset = DirectoryImageDataset(
            Path(config.data.test_dir), image_size=config.data.image_size
        )
        loader = DataLoader(dataset, batch_size=config.data.batch_size, shuffle=False)
        histogram = compute_histogram(
            loader,
            config.evaluation.hue_bins,
            min_chroma=config.evaluation.min_chroma,
        )
        palette = compute_top_colors_modes(
            loader,
            chroma_floor=config.evaluation.min_chroma,
            rng=torch.Generator().manual_seed(config.seed),
        )
        plot_histogram(histogram, title=f"{config.data.name}: test-set hue")
        plot_top_colors_palette(palette, title=f"{config.data.name}: physical ab modes")

## Interpretation and limits

Hue is undefined at zero chroma. Distribution differences describe the input
populations and cannot establish prediction bias or model quality. Record
dataset versions, licenses, exact split manifests, and threshold settings.
No dataset distribution or cross-dataset conclusion is confirmed here.